In [1]:
import pandas as pd
import pyarrow
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import joblib
import boto3

In [2]:
s3_path = "s3://security-data-lake-1.0/analytics/"

df = pd.read_parquet(s3_path)

print(df.shape)
print(df["Label"].value_counts())

(23675873, 9)
Label
Benign                      20746260
DDoS attacks-LOIC-HTTP       1152382
DDOS attack-HOIC              668461
DoS attacks-Hulk              434873
Bot                           282310
Infilteration                 161059
SSH-Bruteforce                117322
DoS attacks-GoldenEye          41455
FTP-BruteForce                 39346
DoS attacks-SlowHTTPTest       19462
DoS attacks-Slowloris          10285
DDOS attack-LOIC-UDP            1730
Brute Force -Web                 611
Brute Force -XSS                 230
SQL Injection                     87
Name: count, dtype: int64


In [3]:
df["Attack"] = (df["Label"] != "Benign").astype(int)

print(df["Attack"].value_counts())

Attack
0    20746260
1     2929613
Name: count, dtype: int64


In [5]:
samples = []

samples.append(
    df[df["Label"] == "Benign"]
      .sample(500000, random_state=42)
)

samples.append(
    df[df["Label"] == "DDoS attacks-LOIC-HTTP"]
      .sample(200000, random_state=42)
)

samples.append(
    df[df["Label"] == "DDOS attack-HOIC"]
      .sample(200000, random_state=42)
)

samples.append(
    df[df["Label"] == "DoS attacks-Hulk"]
      .sample(200000, random_state=42)
)

samples.append(
    df[df["Label"] == "Bot"]
      .sample(200000, random_state=42)
)

rare_classes = [
    "Infilteration",
    "SSH-Bruteforce",
    "DoS attacks-GoldenEye",
    "FTP-BruteForce",
    "DoS attacks-SlowHTTPTest",
    "DoS attacks-Slowloris",
    "DDOS attack-LOIC-UDP",
    "Brute Force -Web",
    "Brute Force -XSS",
    "SQL Injection"
]

for attack in rare_classes:
    samples.append(
        df[df["Label"] == attack]
    )

df_sample = pd.concat(samples)

In [6]:
features = [
    "Dst Port",
    "Protocol",
    "Flow Duration",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts"
]

X = df_sample[features]
y = df_sample["Label"]

In [7]:
le = LabelEncoder()

y = le.fit_transform(df_sample["Label"])

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(np.unique(y)),
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [10]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.98      0.88    100000
           1       1.00      1.00      1.00     40000
           2       0.69      0.61      0.64       122
           3       0.93      0.59      0.72        46
           4       0.67      0.86      0.75     40000
           5       0.90      0.99      0.94       346
           6       0.96      0.99      0.97     40000
           7       0.95      0.82      0.88      8291
           8       0.77      0.57      0.65     40000
           9       0.67      0.39      0.49      3893
          10       0.99      0.98      0.99      2057
          11       0.75      0.90      0.82      7869
          12       0.91      0.27      0.41     32212
          13       1.00      0.47      0.64        17
          14       1.00      1.00      1.00     23465

    accuracy                           0.84    338318
   macro avg       0.87      0.76      0.79    338318
weighted avg       0.85   

In [11]:
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print("Train:", train_score)
print("Test :", test_score)

Train: 0.8411062397793787
Test : 0.8406469652811852


In [12]:
for i, label in enumerate(le.classes_):
    print(i, label)

0 Benign
1 Bot
2 Brute Force -Web
3 Brute Force -XSS
4 DDOS attack-HOIC
5 DDOS attack-LOIC-UDP
6 DDoS attacks-LOIC-HTTP
7 DoS attacks-GoldenEye
8 DoS attacks-Hulk
9 DoS attacks-SlowHTTPTest
10 DoS attacks-Slowloris
11 FTP-BruteForce
12 Infilteration
13 SQL Injection
14 SSH-Bruteforce


In [24]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

print(importance.sort_values("Importance", ascending=False))

         Feature  Importance
0       Dst Port    0.342448
5   Tot Fwd Pkts    0.237011
6   Tot Bwd Pkts    0.144428
4    Flow Pkts/s    0.119781
2  Flow Duration    0.102815
3    Flow Byts/s    0.028081
1       Protocol    0.025435


In [15]:
joblib.dump(model, "xgboost_balanced.pkl")

['xgboost_balanced.pkl']

In [19]:
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [16]:
flow = pd.DataFrame({
    "Dst Port": [22],
    "Protocol": [6],
    "Flow Duration": [500],
    "Flow Byts/s": [20000],
    "Flow Pkts/s": [100],
    "Tot Fwd Pkts": [30],
    "Tot Bwd Pkts": [20]
})

prediction = model.predict(flow)

print(
    "Attack Type:",
    le.inverse_transform(prediction)[0]
)

Attack Type: SSH-Bruteforce


In [20]:
bucket = "security-data-lake-1.0"
s3 = boto3.client("s3")

s3.upload_file("xgboost_balanced.pkl", bucket, "models/xgboost_balanced.pkl")
s3.upload_file("label_encoder.pkl", bucket, "models/label_encoder.pkl")

In [8]:
xgb_model = joblib.load("xgboost_balanced.pkl")
le = joblib.load("label_encoder.pkl")

flow = df.iloc[[0]][features]

prediction = xgb_model.predict(flow)
attack_name = le.inverse_transform(prediction)[0]

print("Predicted attack:", attack_name)
print("Actual attack:", df.iloc[0]["Label"])

Predicted attack: Benign
Actual attack: Benign


In [11]:
sample_df = df.sample(20, random_state=42)

flows = sample_df[features]

predictions = xgb_model.predict(flows)
predicted_names = le.inverse_transform(predictions)

results = pd.DataFrame({
    "Actual": sample_df["Label"].values,
    "Predicted": predicted_names
})

print(results)

                    Actual               Predicted
0                   Benign                  Benign
1         DDOS attack-HOIC        DDOS attack-HOIC
2                   Benign                  Benign
3                   Benign                  Benign
4                   Benign                  Benign
5                   Benign                  Benign
6                   Benign                  Benign
7                   Benign                  Benign
8                   Benign                  Benign
9                   Benign                  Benign
10        DDOS attack-HOIC        DoS attacks-Hulk
11                  Benign                  Benign
12                  Benign                  Benign
13                  Benign                  Benign
14                  Benign                  Benign
15                  Benign                  Benign
16                  Benign                  Benign
17  DDoS attacks-LOIC-HTTP  DDoS attacks-LOIC-HTTP
18                  Benign     